In [ ]:
import sys

from pathlib import Path

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

from config_loaders.dataset_config_loader import DatasetConfig
from datasets import load_dataset, load_from_disk
from utils.data_io import make_dir, save_jsonl
from utils.question_formatter import QuestionFormatter

In [2]:
dataset = "vqav2"
ds_config = DatasetConfig()
ds_paths = ds_config.get_ds_paths(dataset)
ds_paths

{'how_many_people_are': {'imgs': 'data/datasets/vqav2/imgs/how_many_people_are',
  'questions': 'data/datasets/vqav2/q_how_many_people_are.jsonl'},
 'what_is_the_person': {'imgs': 'data/datasets/vqav2/imgs/what_is_the_person',
  'questions': 'data/datasets/vqav2/q_what_is_the_person.jsonl'}}

In [3]:
question_formatter = QuestionFormatter(dataset)

### Downloading and extracting data (takes time)

Skip to next section if arrow files are already present

In [3]:
ds = load_dataset("lmms-lab/VQAv2", split="validation")

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/67 [00:00<?, ?it/s]

In [4]:
ds

Dataset({
    features: ['question_type', 'multiple_choice_answer', 'answers', 'image_id', 'answer_type', 'question_id', 'question', 'image'],
    num_rows: 214354
})

In [9]:
ds[0]

{'question_type': 'none of the above',
 'multiple_choice_answer': 'down',
 'answers': [{'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 1},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 2},
  {'answer': 'at table', 'answer_confidence': 'yes', 'answer_id': 3},
  {'answer': 'skateboard', 'answer_confidence': 'yes', 'answer_id': 4},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 5},
  {'answer': 'table', 'answer_confidence': 'yes', 'answer_id': 6},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 7},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 8},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 9},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 10}],
 'image_id': 262148,
 'answer_type': 'other',
 'question_id': 262148000,
 'question': 'Where is he looking?',
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x512>}

In [6]:
cats_to_keep = [
    "what is the person",
    "how many people are",
]

Filter and save "What is the person..."

In [7]:
vqa_witp = ds.filter(
    lambda entry: entry["question_type"] == "what is the person", num_proc=4
)

Filter (num_proc=4):   0%|          | 0/214354 [00:00<?, ? examples/s]

In [8]:
backup_path = "../data/datasets/vqav2/data_backup/what_is_the_person"
vqa_witp.save_to_disk(backup_path)

Saving the dataset (0/1 shards):   0%|          | 0/900 [00:00<?, ? examples/s]

Filter and save "How many people are..."

In [10]:
vqa_hmpa = ds.filter(
    lambda entry: entry["question_type"] == "how many people are", num_proc=4
)

Filter (num_proc=4):   0%|          | 0/214354 [00:00<?, ? examples/s]

In [11]:
backup_path = "../data/datasets/vqav2/data_backup/how_many_people_are"
vqa_hmpa.save_to_disk(backup_path)

Saving the dataset (0/1 shards):   0%|          | 0/2005 [00:00<?, ? examples/s]

### Process "What is the person..."

In [4]:
c = "what_is_the_person"
backup_path = f"../data/datasets/vqav2/data_backup/{c}"
vqa_witp = load_from_disk(backup_path)
vqa_witp

Dataset({
    features: ['question_type', 'multiple_choice_answer', 'answers', 'image_id', 'answer_type', 'question_id', 'question', 'image'],
    num_rows: 900
})

In [5]:
q_witp = []

img_path_col_witp = [""] * len(vqa_witp)
if "img_path" in vqa_witp:
    vqa_witp = vqa_witp.remove_columns("img_path")

vqa_witp = vqa_witp.add_column("img_path", img_path_col_witp)

out_dir = f"../{ds_paths[c]['imgs']}"
make_dir(out_dir)

for entry in vqa_witp:
    img_path = f"{entry['image_id']}.jpg"
    entry["img_path"] = f"{out_dir}/{img_path}"

    curr_img = entry["image"].convert("RGB")
    curr_img.save(entry["img_path"])
    entry.pop("image")

    q_witp.append(entry)

[Errno 17] File exists: '../data/datasets/vqav2/imgs/what_is_the_person'


In [6]:
new_q_witp = [question_formatter.format(raw_q, c) for raw_q in q_witp]
save_jsonl(new_q_witp, f"../{ds_paths[c]['questions']}")

### Process "How many people are..."

In [7]:
c = "how_many_people_are"
backup_path = f"../data/datasets/vqav2/data_backup/{c}"
vqa_hmpa = load_from_disk(backup_path)
vqa_hmpa

Dataset({
    features: ['question_type', 'multiple_choice_answer', 'answers', 'image_id', 'answer_type', 'question_id', 'question', 'image'],
    num_rows: 2005
})

In [8]:
q_hmpa = []

img_path_col_hmpa = [""] * len(vqa_hmpa)
if "img_path" in vqa_hmpa:
    vqa_hmpa = vqa_hmpa.remove_columns("img_path")

vqa_hmpa = vqa_hmpa.add_column("img_path", img_path_col_hmpa)

out_dir = f"../{ds_paths[c]['imgs']}"
make_dir(out_dir)

for entry in vqa_hmpa:
    img_path = f"{entry['image_id']}.jpg"
    entry["img_path"] = f"{out_dir}/{img_path}"

    curr_img = entry["image"].convert("RGB")
    curr_img.save(entry["img_path"])
    entry.pop("image")

    q_hmpa.append(entry)

[Errno 17] File exists: '../data/datasets/vqav2/imgs/how_many_people_are'


In [9]:
new_q_hmpa = [question_formatter.format(raw_q, c) for raw_q in q_hmpa]
save_jsonl(new_q_hmpa, f"../{ds_paths['how_many_people_are']['questions']}")